In [ ]:
# Install required packages
!pip install -q "transformers>=4.40,<5.0" accelerate torchaudio librosa scikit-learn

# Clone the SOXAN repository for Persian speech models
!git clone -q https://github.com/m3hrdadfi/soxan.git
import sys
sys.path.append("/content/soxan")
from src.models import Wav2Vec2ForSpeechClassification

import torch
print("GPU available:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU available: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/final_project/Shemo_Augmented"

print("Contents of the folder:")
for item in os.listdir(BASE_DIR):
    print(" -", item)

Contents of the folder:
 - shemo_augmented_manifest.csv
 - shemo_augmented


In [ ]:
import glob
import pandas as pd

csv_candidates = glob.glob(f"{BASE_DIR}/*.csv")
print("Found CSV file:", csv_candidates)

csv_path = csv_candidates[0]
df = pd.read_csv(csv_path)
print("DataFrame shape:", df.shape)
df.head()

Found CSV file: ['/content/drive/MyDrive/final_project/Shemo_Augmented/shemo_augmented_manifest.csv']
DataFrame shape: (9464, 6)


,path,original_path,speaker_id,gender,emotion,is_augmented
0,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,13,F,ANGRY,False
1,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,23,F,ANGRY,False
2,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,NEUTRAL,False
3,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,SAD,False
4,/kaggle/input/datasets/mansourehk/shemo-persia...,/kaggle/input/datasets/mansourehk/shemo-persia...,24,F,SAD,False


In [ ]:
def extract_base_id(path_str):
    return os.path.splitext(os.path.basename(path_str))[0]

df["base_id"] = df["path"].apply(extract_base_id)

print("Unique emotion labels in CSV:", df["emotion"].unique())

label_lookup = df.drop_duplicates("base_id").set_index("base_id")[["emotion", "speaker_id", "gender"]].to_dict("index")
print(f"Number of unique base_ids in CSV: {len(label_lookup)}")

Unique emotion labels in CSV: ['ANGRY' 'NEUTRAL' 'SAD' 'HAPPY']
Number of unique base_ids in CSV: 9464


In [ ]:
import re
from collections import Counter

AUDIO_DIR = "/content/drive/MyDrive/final_project/Shemo_Augmented/shemo_augmented/"
audio_files = glob.glob(f"{AUDIO_DIR}/**/*.wav", recursive=True)
print(f"Total audio files: {len(audio_files)}")
print("Sample filenames:", [os.path.basename(f) for f in audio_files[:5]])

def get_base_id_from_audio_filename(filepath):
    filename = os.path.splitext(os.path.basename(filepath))[0]
    return re.sub(r'_aug\d+$', '', filename)

EMOTION_MAP = {
    "ANGRY": "anger",
    "HAPPY": "happiness",
    "HAPPINESS": "happiness",
    "NEUTRAL": "neutral",
    "SAD": "sadness",
    "SADNESS": "sadness",
}

labeled_files = []
unmatched = 0
for f in audio_files:
    base_id = get_base_id_from_audio_filename(f)
    info = label_lookup.get(base_id)
    if info is None:
        unmatched += 1
        continue
    emo = EMOTION_MAP.get(info["emotion"].upper())
    if emo is None:
        continue
    labeled_files.append((f, emo))

print(f"Number of matched and labeled files (4 classes): {len(labeled_files)}")
print(f"Number of unmatched (base_id not found in CSV): {unmatched}")
print(Counter([l for _, l in labeled_files]))

Total audio files: 6727
Sample filenames: ['M32S04_aug1.wav', 'M34S04_aug1.wav', 'M32H02_aug5.wav', 'M31N02_aug1.wav', 'M32S04_aug0.wav']
Number of matched and labeled files (4 classes): 6727
Number of unmatched (base_id not found in CSV): 0
Counter({'anger': 2118, 'neutral': 2056, 'sadness': 1347, 'happiness': 1206})


In [ ]:
from sklearn.model_selection import train_test_split

all_base_ids = list(set(get_base_id_from_audio_filename(f) for f, _ in labeled_files))
train_ids, val_ids = train_test_split(all_base_ids, test_size=0.15, random_state=42)

train_files = [(f, l) for f, l in labeled_files if get_base_id_from_audio_filename(f) in train_ids]
val_files = [(f, l) for f, l in labeled_files if get_base_id_from_audio_filename(f) in val_ids]

print(f"train: {len(train_files)} | val: {len(val_files)}")

train: 5714 | val: 1013


In [ ]:
from transformers import AutoConfig, Wav2Vec2FeatureExtractor

model_name_or_path = "m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition"
label_list = sorted(set(l for _, l in labeled_files))  # ['anger','happiness','neutral','sadness']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
print("classes:", label2id)

config = AutoConfig.from_pretrained(
    model_name_or_path,
    num_labels=len(label_list),
    label2id=label2id,
    id2label=id2label,
)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name_or_path)
target_sampling_rate = feature_extractor.sampling_rate

model = Wav2Vec2ForSpeechClassification.from_pretrained(
    model_name_or_path,
    config=config,
    ignore_mismatched_sizes=True,
)

classes: {'anger': 0, 'happiness': 1, 'neutral': 2, 'sadness': 3}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSpeechClassification were not initialized from the model checkpoint at m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([6, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([6]) in the checkpoint and torch.Size([4]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print(model)

Wav2Vec2ForSpeechClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=Tr

In [ ]:
# Freeze the feature extractor (CNN layers)
model.wav2vec2.feature_extractor._freeze_parameters()

# Unfreeze only the last 2 transformer encoder layers
num_layers_to_unfreeze = 2
total_layers = len(model.wav2vec2.encoder.layers)
for i, layer in enumerate(model.wav2vec2.encoder.layers):
    grad = i >= (total_layers - num_layers_to_unfreeze)
    for p in layer.parameters():
        p.requires_grad = grad

# Count trainable vs total parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} out of {total:,} ({100*trainable/total:.1f}%)")

Trainable parameters: 35,165,316 out of 316,492,420 (11.1%)


In [ ]:
import torch
import torchaudio
from torch.utils.data import Dataset
from functools import partial

class SERDataset(Dataset):
    def __init__(self, pairs, target_sr):
        self.pairs = pairs
        self.target_sr = target_sr

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path, label = self.pairs[idx]
        speech_array, orig_sr = torchaudio.load(path)
        if speech_array.shape[0] > 1:
            speech_array = speech_array.mean(dim=0, keepdim=True)
        speech = torchaudio.transforms.Resample(orig_sr, self.target_sr)(speech_array).squeeze().numpy()
        return {"speech": speech, "label": label2id[label]}

def collate_fn(batch, feature_extractor, sr):
    speeches = [b["speech"] for b in batch]
    labels = torch.tensor([b["label"] for b in batch])
    inputs = feature_extractor(speeches, sampling_rate=sr, return_tensors="pt", padding=True)
    inputs["labels"] = labels
    return inputs

train_dataset = SERDataset(train_files, target_sampling_rate)
val_dataset = SERDataset(val_files, target_sampling_rate)

In [ ]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="/content/phase1_output",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=partial(collate_fn, feature_extractor=feature_extractor, sr=target_sampling_rate),
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.359000,0.321123,0.898322,0.893196
2,0.292200,0.304108,0.901283,0.896057
3,0.368900,0.301281,0.901283,0.896976


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


TrainOutput(global_step=1074, training_loss=0.3906173286491266, metrics={'train_runtime': 4947.0384, 'train_samples_per_second': 3.465, 'train_steps_per_second': 0.217, 'total_flos': 4.0451895759220547e+18, 'train_loss': 0.3906173286491266, 'epoch': 3.0})

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/final_project/shemo_phase1_checkpoint"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Save the model and feature extractor
trainer.save_model(CHECKPOINT_DIR)
feature_extractor.save_pretrained(CHECKPOINT_DIR)
print(f"Phase 1 model saved to: {CHECKPOINT_DIR}")

# Evaluate the model
print(trainer.evaluate())

Phase 1 model saved to: /content/drive/MyDrive/final_project/shemo_phase1_checkpoint


{'eval_loss': 0.3012809455394745, 'eval_accuracy': 0.9012833168805529, 'eval_f1_macro': 0.8969755363969553, 'eval_runtime': 57.8665, 'eval_samples_per_second': 17.506, 'eval_steps_per_second': 4.389, 'epoch': 3.0}
